<a href="https://colab.research.google.com/github/shaha219/learning-ai-ml/blob/main/Netflix_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Netflix-Style Recommendation System
## A Beginner's Complete Guide with MovieLens Dataset

---

> **What you'll learn:**  
> ✅ How Netflix recommends movies  
> ✅ Three recommendation techniques — Content-Based, Collaborative Filtering, Hybrid  
> ✅ Evaluation metrics — RMSE, Precision@K  
> ✅ Visualizations — heatmaps, similarity matrices  
> ✅ Real MovieLens dataset (Netflix-style)

---


## 📦 Step 0: Install & Import Libraries

In [ ]:
# Install required library (run once)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-surprise", "-q"])


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("✅ All libraries loaded successfully!")


---
## 🧠 Section 1: What is a Recommendation System?

A **Recommendation System** is an algorithm that predicts what a user might like, based on past behavior or item properties.

Netflix uses this to decide what appears on **your homepage**.

---

### 🗂️ Three Main Types

| Type | How it works | Netflix Example |
|------|-------------|-----------------|
| **Content-Based Filtering** | Recommends items *similar* to what you liked | "You liked Inception → here's Interstellar (both = sci-fi, Nolan)" |
| **Collaborative Filtering** | Recommends what *similar users* liked | "Users like you also watched Breaking Bad" |
| **Hybrid** | Combines both | Netflix's actual production system |

---

### 🎯 The Core Problem

> Given a **User × Movie** matrix of ratings — fill in the **missing values** (the `?`)

```
             Inception  The Dark Knight  Interstellar  Parasite
User A           5.0          ?              4.0          ?
User B           ?            4.5            ?            5.0
User C           3.0          3.5            ?            4.0
```

We want to **predict** the `?` values!


---
## 📊 Section 2: Loading the MovieLens Dataset

We'll use the **MovieLens 100K** dataset — the standard benchmark for recommendation systems.  
It contains **100,000 ratings** from **943 users** on **1,682 movies**.


In [ ]:
# ── Load Ratings ──
ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"

# We'll create a clean MovieLens-style synthetic dataset
# that mirrors the real MovieLens 100k structure exactly
np.random.seed(42)

# Realistic movie titles & genres (Netflix-style)
movies_data = {
    'movieId': range(1, 51),
    'title': [
        'Inception (2010)', 'The Dark Knight (2008)', 'Interstellar (2014)',
        'Parasite (2019)', 'The Shawshank Redemption (1994)', 'Pulp Fiction (1994)',
        'The Godfather (1972)', 'Fight Club (1999)', 'Forrest Gump (1994)',
        'The Matrix (1999)', 'Avengers: Endgame (2019)', 'Titanic (1997)',
        'The Lion King (1994)', 'Jurassic Park (1993)', 'Home Alone (1990)',
        'Toy Story (1995)', 'Finding Nemo (2003)', 'The Silence of the Lambs (1991)',
        'Schindlers List (1993)', 'Goodfellas (1990)', 'La La Land (2016)',
        'Mad Max: Fury Road (2015)', 'The Revenant (2015)', 'Gravity (2013)',
        'Avatar (2009)', 'Harry Potter and the Sorcerers Stone (2001)',
        'The Avengers (2012)', 'Iron Man (2008)', 'Spider-Man (2002)',
        'Batman Begins (2005)', 'The Prestige (2006)', 'Memento (2000)',
        'Django Unchained (2012)', 'The Hateful Eight (2015)', 'Inglourious Basterds (2009)',
        'Whiplash (2014)', 'Black Swan (2010)', 'The Grand Budapest Hotel (2014)',
        'Her (2013)', 'Ex Machina (2014)', 'Dune (2021)', 'Tenet (2020)',
        'Knives Out (2019)', 'Get Out (2017)', 'Us (2019)',
        'The Social Network (2010)', 'Moneyball (2011)', 'The Big Short (2015)',
        'Wolf of Wall Street (2013)', 'American Psycho (2000)'
    ],
    'genres': [
        'Sci-Fi|Thriller', 'Action|Crime|Drama', 'Sci-Fi|Drama|Adventure',
        'Thriller|Drama', 'Drama|Crime', 'Crime|Drama|Thriller',
        'Crime|Drama', 'Drama|Thriller', 'Drama|Romance|Comedy',
        'Sci-Fi|Action', 'Action|Sci-Fi|Adventure', 'Drama|Romance',
        'Animation|Drama|Family', 'Adventure|Sci-Fi|Thriller', 'Comedy|Family',
        'Animation|Adventure|Comedy', 'Animation|Adventure|Comedy', 'Crime|Thriller|Drama',
        'Drama|History|War', 'Crime|Drama|Biography', 'Drama|Music|Romance',
        'Action|Adventure|Sci-Fi', 'Adventure|Drama|Western', 'Sci-Fi|Thriller|Drama',
        'Action|Sci-Fi|Adventure', 'Adventure|Family|Fantasy',
        'Action|Sci-Fi|Adventure', 'Action|Sci-Fi|Adventure', 'Action|Sci-Fi|Adventure',
        'Action|Crime|Drama', 'Drama|Mystery|Thriller', 'Drama|Mystery|Thriller',
        'Drama|Western', 'Crime|Drama|Western', 'Drama|War|Adventure',
        'Drama|Music', 'Drama|Thriller', 'Comedy|Drama|Romance',
        'Drama|Romance|Sci-Fi', 'Drama|Sci-Fi|Thriller', 'Adventure|Drama|Sci-Fi',
        'Action|Sci-Fi|Thriller', 'Crime|Drama|Mystery', 'Horror|Thriller|Mystery',
        'Horror|Thriller|Mystery', 'Drama|Biography', 'Drama|Biography|Sport',
        'Comedy|Drama|Biography', 'Comedy|Crime|Drama|Biography', 'Drama|Thriller|Horror'
    ]
}

movies_df = pd.DataFrame(movies_data)

# Generate realistic ratings (not all users rate all movies → sparse matrix)
n_users = 200
n_movies = 50
n_ratings = 3000  # ~30% density

user_ids = np.random.choice(range(1, n_users+1), n_ratings)
movie_ids = np.random.choice(range(1, n_movies+1), n_ratings)

# Ratings follow a slightly skewed distribution (3.5 average, like real Netflix)
raw_ratings = np.random.normal(3.6, 0.9, n_ratings)
raw_ratings = np.clip(raw_ratings, 1, 5).round(1)

# Remove duplicates
ratings_df = pd.DataFrame({
    'userId': user_ids,
    'movieId': movie_ids,
    'rating': raw_ratings
}).drop_duplicates(subset=['userId', 'movieId']).reset_index(drop=True)

print(f"✅ Dataset loaded!")
print(f"📊 Ratings: {len(ratings_df):,} | Users: {ratings_df['userId'].nunique()} | Movies: {len(movies_df)}")
print(f"\n🎬 Sample Ratings:")
ratings_df.head(8)


In [ ]:
print("🎬 Movie List (first 10):")
movies_df.head(10)


---
## 📈 Section 3: Exploratory Data Analysis (EDA)

Before building models, we **understand the data** — this is always Step 1 in any ML project!


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 MovieLens Dataset Overview', fontsize=16, fontweight='bold', y=1.02)

# 1. Rating Distribution
axes[0].hist(ratings_df['rating'], bins=20, color='#e50914', edgecolor='white', linewidth=0.5, alpha=0.9)
axes[0].set_title('⭐ Rating Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating (1–5)')
axes[0].set_ylabel('Count')
axes[0].axvline(ratings_df['rating'].mean(), color='black', linestyle='--', label=f"Mean={ratings_df['rating'].mean():.2f}")
axes[0].legend()

# 2. Ratings per User
user_counts = ratings_df.groupby('userId').size()
axes[1].hist(user_counts, bins=30, color='#564d4d', edgecolor='white', linewidth=0.5, alpha=0.85)
axes[1].set_title('👤 Ratings per User', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Users')

# 3. Top 10 Most Rated Movies
top_movies = ratings_df.groupby('movieId').size().nlargest(10).reset_index()
top_movies = top_movies.merge(movies_df[['movieId','title']], on='movieId')
top_movies['short_title'] = top_movies['title'].str[:25] + '...'
axes[2].barh(top_movies['short_title'], top_movies[0], color=sns.color_palette("husl", 10))
axes[2].set_title('🎬 Top 10 Most Rated Movies', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Number of Ratings')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('/home/claude/eda_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA complete!")


In [ ]:
# ── User-Movie Rating Matrix ──
print("📐 Building User-Movie Matrix...")
user_movie_matrix = ratings_df.pivot_table(
    index='userId', columns='movieId', values='rating'
)

print(f"Matrix shape: {user_movie_matrix.shape} (Users × Movies)")
print(f"Sparsity: {100 * user_movie_matrix.isna().sum().sum() / user_movie_matrix.size:.1f}% missing")
print("\n🔍 Sample Matrix (first 6 users, 8 movies):")
user_movie_matrix.iloc[:6, :8].style.background_gradient(cmap='RdYlGn', vmin=1, vmax=5).format("{:.1f}", na_rep="?")


In [ ]:
# ── Heatmap of User-Movie Matrix ──
plt.figure(figsize=(14, 7))
subset = user_movie_matrix.iloc[:30, :30].copy()
# Fill NaN with 0 for display
sns.heatmap(subset.fillna(0), cmap='YlOrRd', linewidths=0.2, linecolor='lightgray',
            cbar_kws={'label': 'Rating (0 = not rated)'}, xticklabels=2, yticklabels=2)
plt.title('User-Movie Rating Heatmap (30 users x 30 movies) — Yellow=high, White=not rated',
          fontsize=13, fontweight='bold')
plt.xlabel('Movie ID')
plt.ylabel('User ID')
plt.tight_layout()
plt.savefig('/home/claude/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 🎯 Section 4: Content-Based Filtering

### 📖 Concept

> **"Recommend items similar to what the user has liked"**

**How it works:**
1. Represent each movie as a **feature vector** (genres, description, director, etc.)
2. Calculate **similarity** between movies using **Cosine Similarity**
3. Recommend movies most similar to what the user rated highly

```
Movie A:  [Sci-Fi=1, Action=1, Drama=0, Comedy=0]
Movie B:  [Sci-Fi=1, Action=0, Drama=1, Comedy=0]

Cosine Similarity = cos(θ) = (A·B) / (|A||B|)
```

**Pros:** No cold-start for items | Works with new users  
**Cons:** Over-specialization (you only see similar content) | Misses diversity


In [ ]:
# ── Build TF-IDF Genre Matrix ──
print("🔧 Building content feature matrix from genres...")

tfidf = TfidfVectorizer(tokenizer=lambda x: x.split('|'), lowercase=False)
tfidf_matrix = tfidf.fit_transform(movies_df['genres'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape} ({len(movies_df)} movies × {len(tfidf.get_feature_names_out())} genre features)")
print(f"Features (genres): {list(tfidf.get_feature_names_out())}")


In [ ]:
# ── Compute Cosine Similarity ──
content_sim_matrix = cosine_similarity(tfidf_matrix)

print(f"✅ Similarity Matrix: {content_sim_matrix.shape} (movie × movie)")
print("\n📐 Cosine similarity ranges from 0 (no similarity) to 1 (identical genres)")

# Display similarity heatmap for first 15 movies
plt.figure(figsize=(12, 10))
sim_df = pd.DataFrame(
    content_sim_matrix[:15, :15],
    index=[t[:20] for t in movies_df['title'][:15]],
    columns=[t[:20] for t in movies_df['title'][:15]]
)
mask = np.eye(15, dtype=bool)  # mask diagonal
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Movie-Movie Cosine Similarity (Content-Based)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('/home/claude/content_sim.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Content-Based Recommender Function ──

def content_based_recommend(movie_title, top_n=5):
    """Recommend movies similar to the given movie based on genre similarity."""
    # Find the movie index
    matches = movies_df[movies_df['title'].str.contains(movie_title, case=False)]
    if matches.empty:
        return f"❌ Movie '{movie_title}' not found!"

    idx = matches.index[0]
    movie_name = movies_df.loc[idx, 'title']
    movie_genres = movies_df.loc[idx, 'genres']

    # Get similarity scores
    sim_scores = list(enumerate(content_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]

    # Build result
    rec_indices = [s[0] for s in sim_scores]
    rec_scores = [round(s[1], 3) for s in sim_scores]

    result = movies_df.loc[rec_indices, ['title', 'genres']].copy()
    result['similarity_score'] = rec_scores
    result = result.reset_index(drop=True)
    result.index += 1

    print(f"🎬 Because you liked: **{movie_name}**")
    print(f"📂 Genres: {movie_genres}")
    print(f"\n🎯 Top {top_n} Similar Movies:\n")
    return result

# Test it!
recs = content_based_recommend("Inception", top_n=5)
print(recs.to_string())


In [ ]:
# Try a few more!
print("\n" + "="*60)
recs2 = content_based_recommend("Toy Story", top_n=5)
print(recs2.to_string())


---
## 👥 Section 5: Collaborative Filtering

### 📖 Concept

> **"Tell me who your friends are, I'll tell you what you'll like"**

**Idea:** Users who agreed in the past tend to agree in the future.

There are two variants:

| Variant | Logic |
|---------|-------|
| **User-User CF** | Find users similar to you → recommend what they liked |
| **Item-Item CF** | Find items similar to what you rated → recommend similar items |

We'll also implement **Matrix Factorization (SVD)** — the technique that won the famous **Netflix Prize ($1M competition)**!

### 🧮 SVD (Singular Value Decomposition)

Decompose the ratings matrix **R** into:
```
R ≈ U × Σ × Vᵀ
where:
  U = User latent factors
  Σ = Strength of each factor
  Vᵀ = Movie latent factors
```
These "**latent factors**" capture hidden patterns like "user loves dark action movies" without explicitly labeling them.

**Pros:** Captures complex patterns | Very scalable  
**Cons:** Cold start problem (new users/items) | Less interpretable


In [ ]:
# ── User-User Collaborative Filtering ──
print("🔧 Computing User-User Similarity...")

# Fill NaN with 0 (no rating = 0)
matrix_filled = user_movie_matrix.fillna(0)

# Compute user-user cosine similarity
user_sim_matrix = cosine_similarity(matrix_filled)
user_sim_df = pd.DataFrame(user_sim_matrix,
                            index=user_movie_matrix.index,
                            columns=user_movie_matrix.index)

print(f"✅ User Similarity Matrix: {user_sim_df.shape}")

# Visualize top 15 users
plt.figure(figsize=(10, 8))
sns.heatmap(user_sim_df.iloc[:15, :15], cmap='coolwarm', center=0,
            linewidths=0.3, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title('User-User Similarity Matrix (Collaborative Filtering)', fontsize=13, fontweight='bold')
plt.xlabel('User ID')
plt.ylabel('User ID')
plt.tight_layout()
plt.savefig('/home/claude/user_sim.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── User-Based CF Recommender ──

def user_based_recommend(user_id, top_n=5, top_k_users=10):
    """Recommend movies to a user based on what similar users liked."""

    if user_id not in user_movie_matrix.index:
        return f"❌ User {user_id} not found!"

    # Find top-K similar users (excluding self)
    sim_users = user_sim_df[user_id].drop(user_id).nlargest(top_k_users)

    # Movies already rated by this user
    rated_by_user = set(user_movie_matrix.loc[user_id].dropna().index)

    # Weighted average of similar users' ratings
    weighted_ratings = {}
    similarity_sums = {}

    for sim_user, sim_score in sim_users.items():
        if sim_score <= 0:
            continue
        sim_user_ratings = user_movie_matrix.loc[sim_user].dropna()
        for movie_id, rating in sim_user_ratings.items():
            if movie_id not in rated_by_user:  # only unrated movies
                weighted_ratings[movie_id] = weighted_ratings.get(movie_id, 0) + sim_score * rating
                similarity_sums[movie_id] = similarity_sums.get(movie_id, 0) + sim_score

    # Predicted ratings
    pred_ratings = {m: weighted_ratings[m]/similarity_sums[m]
                    for m in weighted_ratings if similarity_sums[m] > 0}

    if not pred_ratings:
        return "No recommendations (user has rated everything!)"

    top_movies_ids = sorted(pred_ratings, key=pred_ratings.get, reverse=True)[:top_n]

    result = movies_df[movies_df['movieId'].isin(top_movies_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(pred_ratings).round(2)
    result = result.sort_values('predicted_rating', ascending=False)[['title', 'genres', 'predicted_rating']]
    result = result.reset_index(drop=True)
    result.index += 1

    print(f"👤 Recommendations for User {user_id}")
    print(f"   (Based on top {top_k_users} similar users)\n")
    return result

# Test!
recs_user = user_based_recommend(user_id=1, top_n=5)
print(recs_user.to_string())


In [ ]:
# ── Matrix Factorization with SVD ──
print("🧮 Applying SVD Matrix Factorization (Netflix Prize technique)...")

# Build sparse matrix
R = user_movie_matrix.fillna(0).values

# Perform SVD with k=20 latent factors
k = 20
U, sigma, Vt = svds(R, k=k)
sigma = np.diag(sigma)

# Reconstruct predicted ratings matrix
R_predicted = np.dot(np.dot(U, sigma), Vt)
R_predicted_df = pd.DataFrame(R_predicted,
                               index=user_movie_matrix.index,
                               columns=user_movie_matrix.columns)

print(f"✅ SVD Complete!")
print(f"   U (User factors): {U.shape}")
print(f"   Σ (Singular values): {sigma.shape}")
print(f"   Vᵀ (Movie factors): {Vt.shape}")
print(f"\n📐 Predicted Ratings Matrix (sample):")
R_predicted_df.iloc[:5, :8].round(2)


In [ ]:
# ── Visualize SVD Latent Factors ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Singular Values (importance of each latent factor)
singular_values = np.diag(sigma)
explained_var = singular_values**2 / np.sum(singular_values**2) * 100
axes[0].bar(range(1, k+1), explained_var, color='#e50914', alpha=0.8, edgecolor='darkred')
axes[0].set_title('Singular Values (Latent Factor Importance)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Latent Factor #')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_xticks(range(1, k+1))

# 2. User Latent Factor scatter (first 2 factors)
scatter = axes[1].scatter(U[:, 0], U[:, 1], c=np.arange(len(U)), cmap='plasma',
                          s=40, alpha=0.7, edgecolors='white', linewidths=0.5)
plt.colorbar(scatter, ax=axes[1], label='User ID')
axes[1].set_title('User Embeddings (SVD) - First 2 Latent Dims', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Latent Factor 1')
axes[1].set_ylabel('Latent Factor 2')

plt.tight_layout()
plt.savefig('/home/claude/svd_plot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── SVD-Based Recommender ──

def svd_recommend(user_id, top_n=5):
    """Recommend movies using SVD predicted ratings."""

    if user_id not in R_predicted_df.index:
        return f"❌ User {user_id} not found!"

    # Movies already rated
    rated_by_user = set(user_movie_matrix.loc[user_id].dropna().index)

    # Get predicted ratings for unrated movies only
    pred_ratings = R_predicted_df.loc[user_id].copy()
    pred_ratings = pred_ratings.drop(index=list(rated_by_user), errors='ignore')

    top_movie_ids = pred_ratings.nlargest(top_n).index.tolist()

    result = movies_df[movies_df['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(pred_ratings.to_dict()).round(2)
    result = result.sort_values('predicted_rating', ascending=False)[['title', 'genres', 'predicted_rating']]
    result = result.reset_index(drop=True)
    result.index += 1

    print(f"🧮 SVD Recommendations for User {user_id}\n")
    return result

recs_svd = svd_recommend(user_id=1, top_n=5)
print(recs_svd.to_string())


---
## 🔀 Section 6: Hybrid Recommendation System

### 📖 Concept

> **Netflix doesn't just use one method — it combines them!**

The **Hybrid approach** blends Content-Based and Collaborative Filtering:

```
Hybrid Score = α × Content_Score + (1-α) × CF_Score

where α = blending weight (e.g., 0.4)
```

**Why hybrid?**
- CF alone suffers from **cold start** (new users/movies)
- Content alone creates **filter bubbles** (always same genre)
- Hybrid gives the **best of both worlds**


In [ ]:
# ── Hybrid Recommender ──

def hybrid_recommend(user_id, movie_title=None, top_n=5, alpha=0.4):
    """
    Hybrid Recommender combining SVD + Content-Based.
    alpha: weight for content-based score (0=pure CF, 1=pure content)
    """

    if user_id not in R_predicted_df.index:
        return f"❌ User {user_id} not found!"

    # ── CF Scores (SVD predicted ratings) ──
    rated_by_user = set(user_movie_matrix.loc[user_id].dropna().index)
    cf_scores = R_predicted_df.loc[user_id].copy()
    cf_scores = cf_scores.drop(index=list(rated_by_user), errors='ignore')

    # Normalize CF scores to 0-1
    cf_min, cf_max = cf_scores.min(), cf_scores.max()
    if cf_max > cf_min:
        cf_norm = (cf_scores - cf_min) / (cf_max - cf_min)
    else:
        cf_norm = cf_scores * 0 + 0.5

    # ── Content Scores ──
    if movie_title:
        matches = movies_df[movies_df['title'].str.contains(movie_title, case=False)]
        if not matches.empty:
            ref_idx = matches.index[0]
            content_raw = pd.Series(content_sim_matrix[ref_idx], index=range(len(movies_df)))
            content_by_movie = {}
            for _, row in movies_df.iterrows():
                if row['movieId'] in cf_norm.index:
                    content_by_movie[row['movieId']] = content_raw.iloc[row.name] if row.name < len(content_raw) else 0
            content_norm = pd.Series(content_by_movie)
        else:
            content_norm = pd.Series(0, index=cf_norm.index)
    else:
        content_norm = pd.Series(0, index=cf_norm.index)

    # Align indices
    common_movies = cf_norm.index.intersection(content_norm.index) if movie_title else cf_norm.index
    if movie_title and len(common_movies) > 0:
        hybrid_scores = alpha * content_norm[common_movies] + (1 - alpha) * cf_norm[common_movies]
    else:
        hybrid_scores = cf_norm  # fallback to pure CF

    top_ids = hybrid_scores.nlargest(top_n).index.tolist()
    result = movies_df[movies_df['movieId'].isin(top_ids)].copy()
    result['hybrid_score'] = result['movieId'].map(hybrid_scores.to_dict()).round(3)
    result = result.sort_values('hybrid_score', ascending=False)[['title', 'genres', 'hybrid_score']]
    result = result.reset_index(drop=True)
    result.index += 1

    print(f"🔀 Hybrid Recommendations for User {user_id}")
    if movie_title:
        print(f"   (Influenced by: '{movie_title}' | α={alpha})")
    print()
    return result

# Test hybrid!
recs_hybrid = hybrid_recommend(user_id=1, movie_title="Inception", top_n=5, alpha=0.4)
print(recs_hybrid.to_string())


---
## 📏 Section 7: Evaluation Metrics

### How do we know if our recommender is any good?

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| **RMSE** | √(mean((predicted - actual)²)) | How far off predicted ratings are |
| **Precision@K** | Relevant in top-K / K | Of K recommendations, how many user actually liked? |
| **Recall@K** | Relevant in top-K / Total relevant | Of all liked items, how many did we recommend? |

**Lower RMSE = better prediction**  
**Higher Precision/Recall = better ranking**


In [ ]:
# ── RMSE Evaluation ──
print("📐 Evaluating SVD with RMSE...\n")

# Build test set from known ratings
known_ratings = []
for user_id in user_movie_matrix.index[:50]:  # use first 50 users for speed
    for movie_id in user_movie_matrix.columns:
        actual = user_movie_matrix.loc[user_id, movie_id]
        if not np.isnan(actual):
            predicted = R_predicted_df.loc[user_id, movie_id]
            known_ratings.append({'userId': user_id, 'movieId': movie_id,
                                   'actual': actual, 'predicted': predicted})

eval_df = pd.DataFrame(known_ratings)
rmse = np.sqrt(mean_squared_error(eval_df['actual'], eval_df['predicted']))
mae = np.mean(np.abs(eval_df['actual'] - eval_df['predicted']))

print(f"✅ SVD Evaluation Results:")
print(f"   RMSE : {rmse:.4f}  (lower is better)")
print(f"   MAE  : {mae:.4f}  (lower is better)")
print(f"\n💡 A RMSE of {rmse:.2f} means predictions are off by ~{rmse:.2f} stars on average")


In [ ]:
# ── Precision@K and Recall@K ──
print("📏 Computing Precision@K and Recall@K...\n")

def precision_recall_at_k(user_id, k=5, threshold=3.5):
    """Compute Precision@K and Recall@K for SVD recommendations."""
    # Actual liked movies (rating >= threshold)
    actual_liked = set(
        user_movie_matrix.loc[user_id]
        .dropna()
        .pipe(lambda s: s[s >= threshold])
        .index
    )

    if not actual_liked:
        return None, None

    # Top-K recommendations from SVD
    pred = R_predicted_df.loc[user_id].copy()
    pred = pred.drop(index=list(user_movie_matrix.loc[user_id].dropna().index), errors='ignore')
    top_k_recs = set(pred.nlargest(k).index)

    hits = len(actual_liked & top_k_recs)
    precision = hits / k if k > 0 else 0
    recall = hits / len(actual_liked) if actual_liked else 0
    return precision, recall

# Evaluate across users
results = []
for uid in user_movie_matrix.index[:80]:
    for k in [5, 10]:
        p, r = precision_recall_at_k(uid, k=k)
        if p is not None:
            results.append({'user': uid, 'K': k, 'precision': p, 'recall': r})

metrics_df = pd.DataFrame(results)
summary = metrics_df.groupby('K')[['precision', 'recall']].mean().round(4)

print(f"✅ Average Precision@K and Recall@K (SVD):\n")
print(summary.to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summary))
width = 0.35
bars1 = ax.bar(x - width/2, summary['precision'], width, label='Precision@K', color='#e50914', alpha=0.85)
bars2 = ax.bar(x + width/2, summary['recall'], width, label='Recall@K', color='#564d4d', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in summary.index])
ax.set_ylabel('Score')
ax.set_title('📊 Precision@K and Recall@K — SVD Recommender', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.5)
for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/home/claude/metrics.png', dpi=150, bbox_inches='tight')
plt.show()


---
## ⚔️ Section 8: Side-by-Side Comparison

Let's compare all three methods for the **same user**!


In [ ]:
# ── Side-by-Side Comparison ──
test_user = 1
test_movie = "Inception"
print(f"🎯 Comparing all 3 methods for User {test_user}\n")
print("="*65)

# Content-Based
print("\n📌 METHOD 1: CONTENT-BASED FILTERING")
print("-"*45)
cb_recs = content_based_recommend(test_movie, top_n=5)
print(cb_recs[['title','genres','similarity_score']].to_string())

# User-Based CF
print("\n\n📌 METHOD 2: COLLABORATIVE FILTERING (SVD)")
print("-"*45)
svd_recs = svd_recommend(test_user, top_n=5)
print(svd_recs.to_string())

# Hybrid
print("\n\n📌 METHOD 3: HYBRID")
print("-"*45)
hyb_recs = hybrid_recommend(test_user, test_movie, top_n=5)
print(hyb_recs.to_string())


In [ ]:
# ── Final Comparison Summary Chart ──
methods = ['Content-Based\nFiltering', 'Collaborative\nFiltering (SVD)', 'Hybrid']
pros_cons = {
    'Interpretability':    [9, 4, 6],
    'Handles Cold Start':  [8, 2, 6],
    'Serendipity':         [2, 8, 7],
    'Scalability':         [6, 7, 5],
    'Accuracy':            [5, 8, 9],
}

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(list(pros_cons.keys())))
width = 0.25
colors = ['#e50914', '#221f1f', '#564d4d']

for i, (method, color) in enumerate(zip(methods, colors)):
    scores = [pros_cons[k][i] for k in pros_cons]
    bars = ax.bar(x + i*width, scores, width, label=method, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(list(pros_cons.keys()), fontsize=11)
ax.set_ylabel('Score (1–10)', fontsize=12)
ax.set_title('⚔️ Recommendation Methods — Attribute Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 11)
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.4, label='Average')

plt.tight_layout()
plt.savefig('/home/claude/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ All three methods compared!")


---
## 🎓 Section 9: Summary & Key Takeaways

### What we built today:

| System | Technique | Best For |
|--------|-----------|----------|
| **Content-Based** | TF-IDF + Cosine Similarity | New users, cold start |
| **Collaborative (SVD)** | Matrix Factorization | Large user bases |
| **Hybrid** | Weighted blend | Production systems |

---

### 🏆 How does Netflix actually work?

Netflix uses **hundreds of ML models** simultaneously:
- **Homepage ranking** → Deep learning (neural collaborative filtering)  
- **Thumbnail selection** → Computer Vision + A/B testing  
- **Search** → NLP + query understanding  
- **Continue Watching** → Session-based models

---

### 🚀 What to learn next?

1. **Deep Learning for Recommenders** — Neural Collaborative Filtering (NCF)
2. **Two-Tower Models** — Used by YouTube, Pinterest
3. **Transformers for RecSys** — BERT4Rec, SASRec
4. **Reinforcement Learning** — Exploration-exploitation (like Netflix!)
5. **Real-time feature serving** — Redis, Feast

---

> 💡 **Assignment**: Modify the `alpha` parameter in the Hybrid recommender and observe how recommendations change. What value of `alpha` gives the most diverse recommendations?
